<div dir="rtl">

# חיזוי עלות תביעות בביטוח רכב

המחברת הראשית של הפרויקט, לפי סדר סעיפי המטלה.

מחברות החקירה (cleaning_driver, daniel_prepare, target_eda) נמצאות ב-repo ולא משתנות.

## הכנה: ספריות וגרסאות

RANDOM_STATE מוגדר פעם אחת כאן, ומשמש בכל מקום שיש בו אקראיות. את הגרסאות שמודפסות נעתיק ל-requirements.txt, כי model.pkl נטען רק באותה גרסה של scikit-learn שבה הוא נשמר.

</div>

In [1]:
# ייבוא ספריות, קבוע אקראיות אחד לכל הפרויקט ובדיקת גרסאות
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.16
pandas: 3.1.0
numpy: 2.5.5
scikit-learn: 1.9.1


<div dir="rtl">

## טעינת הדאטה ומבט ראשון

הדאטה כולל 45,000 נהגים ו-30 עמודות. מציגים חמש שורות אקראיות ולא את הראשונות, כי הקובץ עלול להיות מסודר לפי משהו.

</div>

In [2]:
# טעינת הדאטה והצגת חמש שורות אקראיות

df = pd.read_csv("dataset.csv")
print(df.shape)
df.sample(5, random_state=RANDOM_STATE)

(45000, 30)


,customer_id_hash,agent_code,country,area_type,region_density,winter_severity,driver_age,gender,marital_status,license_years,...,annual_km_estimated,pct_intercity,pct_night_driving,parking_type,current_premium,claim_count,at_fault,days_to_settlement,police_report_filed,annual_claim_cost
37979,0ebf89f0,A020,FR,suburban,806,3.0,63,M,married,36,...,66500.0,10,45,garage,4270,0,NaN,NaN,NaN,0
9911,e7bedf11,A143,US,rural,24,0.0,38,M,divorced,20,...,17500.0,15,5,garage,1860,0,NaN,NaN,NaN,0
43386,9ff342a8,A060,FR,suburban,1115,3.8,84,M,married,66,...,3000.0,0,10,garage,1950,0,NaN,NaN,NaN,0
13822,b66c36b7,A004,DE,urban,10949,6.6,48,M,divorced,30,...,NaN,20,30,lot,13810,0,NaN,NaN,NaN,0
44810,30fbec1d,A004,US,rural,100,6.4,80,M,married,62,...,15500.0,55,10,garage,5160,0,NaN,NaN,NaN,0


<div dir="rtl">

בודקים איזה סוג נתונים pandas נתן לכל עמודה. עמודה שאמורה להיות מספרית ונקראה כטקסט מרמזת על ערכים מלוכלכים.

</div>

In [3]:
# סוג הנתונים בכל עמודה, כדי לזהות עמודות מספר שנקראו כטקסט
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id_hash       45000 non-null  str    
 1   agent_code             45000 non-null  str    
 2   country                45000 non-null  str    
 3   area_type              45000 non-null  str    
 4   region_density         45000 non-null  int64  
 5   winter_severity        45000 non-null  float64
 6   driver_age             45000 non-null  int64  
 7   gender                 45000 non-null  str    
 8   marital_status         45000 non-null  str    
 9   license_years          45000 non-null  int64  
 10  prior_claims_5y        41858 non-null  float64
 11  no_claims_years        45000 non-null  int64  
 12  violations_3y          45000 non-null  int64  
 13  vehicle_type           45000 non-null  str    
 14  vehicle_age            45000 non-null  int64  
 15  engine_hp    

<div dir="rtl">

ערכים חסרים כפי שהם בקובץ. זו לא התמונה המלאה, כי חלק מהחוסרים מוסווים כמספרים, וכך נראה בתא הבא.

</div>

In [4]:
# כמות הערכים החסרים בכל עמודה, רק בעמודות שיש בהן חוסרים
missing = df.isna().sum()
missing[missing > 0]

prior_claims_5y         3142
annual_km_estimated     4311
at_fault               29207
days_to_settlement     29207
police_report_filed    29207
dtype: int64

<div dir="rtl">

סטטיסטיקה של העמודות המספריות. המינימום והמקסימום חושפים ערכים לא הגיוניים, שחוקי הניקוי יטפלו בהם בהמשך.

</div>

In [5]:
# סטטיסטיקה של עמודות המספר, כדי לזהות ערכים חריגים ולא הגיוניים
df.describe().T

,count,mean,std,min,25%,50%,75%,max
region_density,45000.0,1886.716311,2161.133202,4.0,498.0,1151.0,2593.0,20000.0
winter_severity,45000.0,3.459389,2.214855,0.0,1.7,3.3,5.0,10.0
driver_age,45000.0,48.010511,16.613355,18.0,35.0,47.0,60.0,90.0
license_years,45000.0,28.853867,16.866459,0.0,15.0,28.0,41.0,95.0
prior_claims_5y,41858.0,1.039706,1.384058,0.0,0.0,1.0,2.0,6.0
no_claims_years,45000.0,6.750933,8.101940,0.0,1.0,3.0,10.0,30.0
violations_3y,45000.0,0.416467,0.717039,0.0,0.0,0.0,1.0,6.0
vehicle_age,45000.0,7.638844,5.313188,0.0,4.0,6.0,10.0,25.0
engine_hp,45000.0,143.920800,66.350986,12.0,94.0,137.0,178.0,650.0
safety_features_count,45000.0,3.983378,2.028846,0.0,2.0,4.0,6.0,8.0


<div dir="rtl">

## 2.2 מניעת Data Leakage

המודל חוזה עלות תביעות של נהג חדש בתחילת השנה. מותר להשתמש רק בשדות שידועים לפני חתימת החוזה.

בשלב זה בודקים כל עמודה חשודה, ומוכיחים בנתונים אם היא מדליפה מידע מהעתיד.

</div>

<div dir="rtl">

### מספר התביעות: claim_count

אם מספר התביעות השנה גדול מאפס בדיוק כשיש עלות, העמודה חושפת את התשובה.

</div>

In [6]:
# בדיקה האם מספר התביעות חופף בדיוק לעלות התביעות
pd.crosstab(df["claim_count"] > 0, df["annual_claim_cost"] > 0,
            rownames=["claim_count > 0"], colnames=["cost > 0"])

cost > 0,False,True
claim_count > 0,,
False,29207,0
True,0,15793


<div dir="rtl">

### פרטי התאונה: at_fault, days_to_settlement, police_report_filed

פרטי התאונה קיימים רק כשהייתה תביעה. בודקים אם גם החוסר עצמו חושף את התשובה.

</div>

In [7]:
# בדיקה האם שלוש עמודות התאונה ריקות בדיוק כשאין תביעה
for col in ["at_fault", "days_to_settlement", "police_report_filed"]:
    print(col)
    print(pd.crosstab(df[col].isna(), df["annual_claim_cost"] > 0,
                      rownames=["missing"], colnames=["cost > 0"]), "\n")

at_fault
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 

days_to_settlement
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 

police_report_filed
cost > 0  False  True 
missing               
False         0  15793
True      29207      0 



<div dir="rtl">

### הפרמיה: current_premium

לפי מילון השדות זו הפרמיה ששולמה בסוף השנה. בודקים אם היא קשורה לתביעות של אותה שנה.

</div>

In [8]:
# בדיקת הקשר בין הפרמיה של סוף השנה לבין עלות התביעות
print("Correlation with target:", round(df["current_premium"].corr(df["annual_claim_cost"]), 3))
df.groupby(df["annual_claim_cost"] > 0)["current_premium"].mean()

Correlation with target: 0.577


annual_claim_cost
False    3188.265484
True     6214.693852
Name: current_premium, dtype: float64

<div dir="rtl">

### שנים ללא תביעה: no_claims_years

אם השדה היה מתעדכן בסוף השנה, נהגים שאשמים בתאונה השנה היו מופיעים כמעט כולם עם 0.

</div>

In [9]:
# בדיקה האם שנות הנהיגה ללא תביעה עודכנו בסוף השנה
df.groupby(df["at_fault"].fillna(-1))["no_claims_years"].apply(lambda s: (s == 0).mean())

at_fault
-1.0    0.146506
 0.0    0.264279
 1.0    0.347816
Name: no_claims_years, dtype: float64

<div dir="rtl">

### מזהים: customer_id_hash ו-agent_code

עמודות מנהליות. בודקים אם המזהה ייחודי וכמה סוכנים יש. את ההשפעה של הסוכן בדקנו בבדיקת ערבוב במחברת cleaning_driver.

</div>

In [10]:
# בדיקת העמודות המנהליות מזהה לקוח וקוד סוכן
print("customer_id_hash unique:", df["customer_id_hash"].is_unique)
print("agent_code values:", df["agent_code"].nunique())

customer_id_hash unique: True
agent_code values: 300


<div dir="rtl">

### מסקנות סעיף 2.2

עמודות Leakage שיוסרו: claim_count, at_fault, days_to_settlement, police_report_filed, current_premium. כולן נוצרות במהלך שנת הביטוח או בסופה, ולכן הן לא ידועות ביום החתימה.

עמודות שיוסרו כי אין להן ערך לחיזוי: customer_id_hash (מזהה ייחודי) ו-agent_code (בדיקת ערבוב במחברת cleaning_driver הראתה שהפיזור בין 300 הסוכנים, 807, לא גדול מהפיזור המקרי, 803).

עמודה שנבדקה ונמצאה תקינה: no_claims_years. רק 35% מהאשמים בתאונה השנה מופיעים עם 0, ולכן השדה לא עודכן בסוף השנה.

</div>

In [11]:
# רשימות העמודות שיוסרו מהמודל: דליפת מידע, ומזהים בלי ערך לחיזוי
LEAKAGE_COLS = ["claim_count", "at_fault", "days_to_settlement", "police_report_filed", "current_premium"]
ID_COLS = ["customer_id_hash", "agent_code"]

<div dir="rtl">

## 3.1 ניקוי הדאטה ופונקציית prepare_data

הבודק מריץ את prepare_data על קובץ שלא ראינו. לכן היא חייבת להיות סט של חוקים קבועים, שרצים בלי שגיאה גם על שורה אחת וגם על אלפי נהגים.

ערך לא חוקי הופך ל-NaN, ואף שורה לא נמחקת. את ההשלמה של ערכים חסרים עושה ה-Pipeline, כדי שבכל fold היא תלמד רק מה-train.

### פונקציות עזר

to_num ממירה עמודות מספר, ו-clean_text מאחדת עמודות טקסט. שתיהן עובדות גם על עמודה שכולה ריקה, שאותה pandas קורא כמספר.

</div>

In [12]:
# פונקציות עזר לניקוי: אחת לעמודות מספר ואחת לעמודות טקסט, שעובדות גם על עמודה ריקה לגמרי
def to_num(s):
    # המרה למספר אחרי הסרת פסיקים ורווחים, וערך אינסופי הופך לחסר
    out = pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
    return out.replace([np.inf, -np.inf], np.nan)

def clean_text(s, case="lower"):
    # הסרת רווחים ואחידות אותיות, וערך חסר או מחרוזת ריקה הופכים לחסר
    out = s.astype(str).str.strip()
    out = out.str.upper() if case == "upper" else out.str.lower()
    return out.where(s.notna() & (out != ""))

In [13]:
# הדגמה קצרה של שתי הפונקציות על ערכים מלוכלכים ועל עמודה שכולה ריקה
print(to_num(pd.Series(["1,200", " 7 ", "inf", "", None, "abc"])).tolist())
print(clean_text(pd.Series([" us", "US ", "", None]), case="upper").tolist())
print(to_num(pd.Series([np.nan])).tolist(), clean_text(pd.Series([np.nan])).tolist())

[1200.0, 7.0, nan, nan, nan, nan]
['US', 'US', nan, nan]
[nan] [nan]


<div dir="rtl">

### ניקוי עמודות הנהג, ההיסטוריה והגאוגרפיה

פונקציות העזר עובדות. עכשיו משתמשים בהן לניקוי עמודות A עד M. את החוקים קבע ליעד במחברת cleaning_driver, ושם נמצא גם הנימוק לכל חוק.

</div>

In [14]:
# פונקציית הניקוי הסופית של עמודות הנהג ההיסטוריה והגאוגרפיה
def clean_driver_cols(df):
    df = df.copy()

    # מדינה וסוג אזור: איחוד כתיבה, וערך חסר או ריק נשאר חסר
    df["country"] = clean_text(df["country"], case="upper")
    df["area_type"] = clean_text(df["area_type"], case="lower")

    # צפיפות: המרה למספר, וערך שאינו חיובי הופך לחסר
    df["region_density"] = to_num(df["region_density"])
    df.loc[df["region_density"] <= 0, "region_density"] = np.nan

    # חומרת חורף: ערך מחוץ לטווח אפס עד עשר הופך לחסר
    df["winter_severity"] = to_num(df["winter_severity"])
    df.loc[(df["winter_severity"] < 0) | (df["winter_severity"] > 10), "winter_severity"] = np.nan

    # גיל: גיל מתחת לשש עשרה או מעל מאה הופך לחסר
    df["driver_age"] = to_num(df["driver_age"])
    df.loc[(df["driver_age"] < 16) | (df["driver_age"] > 100), "driver_age"] = np.nan

    # מגדר: מיפוי לשני ערכים תקניים, וערך לא מוכר הופך לחסר
    gender_map = {"m": "M", "male": "M", "f": "F", "female": "F"}
    df["gender"] = clean_text(df["gender"], case="lower").map(gender_map)

    # מצב משפחתי: רק ארבעת הערכים החוקיים נשארים, וכל ערך אחר הופך לחסר
    valid_marital = ["single", "married", "divorced", "widowed"]
    df["marital_status"] = clean_text(df["marital_status"], case="lower")
    df.loc[~df["marital_status"].isin(valid_marital), "marital_status"] = np.nan

    # ותק רישיון: ערך שלילי הופך לחסר, וגם ותק שמשמעותו קבלת רישיון לפני גיל שש עשרה
    df["license_years"] = to_num(df["license_years"])
    df.loc[df["license_years"] < 0, "license_years"] = np.nan
    df.loc[df["driver_age"] - df["license_years"] < 16, "license_years"] = np.nan

    # תביעות בעבר: ערך שלילי הופך לחסר וערך מעל שש הופך לשש
    df["prior_claims_5y"] = to_num(df["prior_claims_5y"])
    df.loc[df["prior_claims_5y"] < 0, "prior_claims_5y"] = np.nan
    df["prior_claims_5y"] = df["prior_claims_5y"].clip(upper=6)

    # נהג עם אפס שנות ותק לא יכול היה לצבור תביעות בעבר ולכן ערך חסר אצלו הופך לאפס
    df.loc[(df["license_years"] == 0) & (df["prior_claims_5y"].isna()), "prior_claims_5y"] = 0

    # שנים ללא תביעה: ערך שלילי או גדול מוותק הרישיון הופך לחסר וערך מעל שלושים הופך לשלושים
    df["no_claims_years"] = to_num(df["no_claims_years"])
    df.loc[df["no_claims_years"] < 0, "no_claims_years"] = np.nan
    df.loc[df["no_claims_years"] > df["license_years"], "no_claims_years"] = np.nan
    df["no_claims_years"] = df["no_claims_years"].clip(upper=30)

    # דוחות תנועה: ערך שלילי הופך לחסר
    df["violations_3y"] = to_num(df["violations_3y"])
    df.loc[df["violations_3y"] < 0, "violations_3y"] = np.nan

    return df

In [15]:
# השוואת ערכים חסרים בעמודות הנהג לפני הניקוי ואחריו
driver_cols = ["country", "area_type", "region_density", "winter_severity", "driver_age", "gender",
               "marital_status", "license_years", "prior_claims_5y", "no_claims_years", "violations_3y"]
driver_clean = clean_driver_cols(df)
print(pd.DataFrame({"before": df[driver_cols].isna().sum(), "after": driver_clean[driver_cols].isna().sum()}))
print(driver_clean["gender"].value_counts(dropna=False))

                 before  after
country               0      0
area_type             0      0
region_density        0      0
winter_severity       0      0
driver_age            0      0
gender                0      0
marital_status        0      0
license_years         0    153
prior_claims_5y    3142   1955
no_claims_years       0      0
violations_3y         0      0
gender
M    24238
F    20762
Name: count, dtype: int64


<div dir="rtl">

אחרי ניקוי עמודות הנהג נשארים ערכים חסרים בשתי עמודות:

license_years: 153 נהגים, שאצלם הגיל פחות הוותק קטן מ-16. נתון כזה לא אפשרי, ולכן הוא הופך לחסר.

prior_claims_5y: ירד מ-3,142 ל-1,955. 1,187 נהגים עם 0 שנות ותק קיבלו 0 תביעות קודמות. זה מילוי לוגי שתלוי רק באותה שורה, ולכן הוא מותר בתוך prepare_data, בניגוד למילוי בממוצע.

מגדר: שמונה צורות כתיבה אוחדו ל-M ו-F, בלי ערכים חסרים.

</div>

<div dir="rtl">

### ניקוי עמודות הרכב והשימוש

אותו עיקרון כמו בעמודות הנהג, הפעם לעמודות N עד X. החוקים נקבעו במחברת daniel_prepare, ושם נמצא גם הנימוק לכל חוק.

</div>

In [16]:
# פונקציית הניקוי הסופית של עמודות הרכב והשימוש
def clean_vehicle_cols(df):
    df = df.copy()

    # סוג רכב: איחוד כתיבה, וערך שאינו אחד מששת הסוגים החוקיים הופך לחסר
    valid_vehicle = ["small", "sedan", "suv", "sports", "pickup", "motorcycle"]
    df["vehicle_type"] = clean_text(df["vehicle_type"], case="lower")
    df.loc[~df["vehicle_type"].isin(valid_vehicle), "vehicle_type"] = np.nan

    # גיל רכב: ערך שלילי הופך לחסר, וערך מעל עשרים וחמש הופך לעשרים וחמש
    df["vehicle_age"] = to_num(df["vehicle_age"])
    df.loc[df["vehicle_age"] < 0, "vehicle_age"] = np.nan
    df["vehicle_age"] = df["vehicle_age"].clip(upper=25)

    # עוצמת מנוע: ערך שאינו חיובי הופך לחסר
    df["engine_hp"] = to_num(df["engine_hp"])
    df.loc[df["engine_hp"] <= 0, "engine_hp"] = np.nan

    # שווי רכב: הפסיקים מוסרים בפונקציית העזר, וערך שאינו חיובי הופך לחסר
    df["vehicle_value"] = to_num(df["vehicle_value"])
    df.loc[df["vehicle_value"] <= 0, "vehicle_value"] = np.nan

    # מערכות בטיחות: ערך מחוץ לטווח אפס עד שמונה הופך לחסר
    df["safety_features_count"] = to_num(df["safety_features_count"])
    df.loc[(df["safety_features_count"] < 0) | (df["safety_features_count"] > 8), "safety_features_count"] = np.nan

    # טלמטיקה: ערך שאינו אפס או אחד הופך לחסר
    df["has_telematics"] = to_num(df["has_telematics"])
    df.loc[~df["has_telematics"].isin([0, 1]), "has_telematics"] = np.nan

    # סוג שימוש: איחוד כתיבה, וערך שאינו אחד מחמשת הסוגים החוקיים הופך לחסר
    valid_usage = ["commute", "admin_work", "leisure", "professional", "delivery"]
    df["usage_type"] = clean_text(df["usage_type"], case="lower")
    df.loc[~df["usage_type"].isin(valid_usage), "usage_type"] = np.nan

    # קילומטרים: ערך שאינו חיובי או מעל מאתיים אלף הופך לחסר, כולל הערך המוסווה של תשע מאות תשעים ותשעה אלף
    df["annual_km_estimated"] = to_num(df["annual_km_estimated"])
    df.loc[(df["annual_km_estimated"] <= 0) | (df["annual_km_estimated"] > 200000), "annual_km_estimated"] = np.nan

    # אחוז נסיעה בין עירונית ואחוז נהיגה בלילה: ערך מחוץ לטווח אפס עד מאה הופך לחסר, כל עמודה בנפרד
    for c in ["pct_intercity", "pct_night_driving"]:
        df[c] = to_num(df[c])
        df.loc[(df[c] < 0) | (df[c] > 100), c] = np.nan

    # סוג חניה: איחוד כתיבה, וערך שאינו אחד משלושת הסוגים החוקיים הופך לחסר
    valid_parking = ["garage", "street", "lot"]
    df["parking_type"] = clean_text(df["parking_type"], case="lower")
    df.loc[~df["parking_type"].isin(valid_parking), "parking_type"] = np.nan

    return df

In [17]:
# השוואת ערכים חסרים בעמודות הרכב לפני הניקוי ואחריו, וסוג הנתונים של שווי הרכב
vehicle_cols = ["vehicle_type", "vehicle_age", "engine_hp", "vehicle_value", "safety_features_count", "has_telematics",
                "usage_type", "annual_km_estimated", "pct_intercity", "pct_night_driving", "parking_type"]
vehicle_clean = clean_vehicle_cols(df)
print(pd.DataFrame({"before": df[vehicle_cols].isna().sum(), "after": vehicle_clean[vehicle_cols].isna().sum()}))
print("vehicle_value dtype before:", df["vehicle_value"].dtype, "| after:", vehicle_clean["vehicle_value"].dtype)

                       before  after
vehicle_type                0      0
vehicle_age                 0      0
engine_hp                   0      0
vehicle_value               0      0
safety_features_count       0      0
has_telematics              0    636
usage_type                  0      0
annual_km_estimated      4311   5697
pct_intercity               0      0
pct_night_driving           0      0
parking_type                0      0
vehicle_value dtype before: str | after: float64


<div dir="rtl">

אחרי ניקוי עמודות הרכב נשארים ערכים חסרים בשתי עמודות:

has_telematics: 636 נהגים עם הערך 2, שכנראה מסמן "לא ידוע". הערך הופך לחסר, כי עמודה בינארית יכולה לקבל רק 0 או 1.

annual_km_estimated: עלה מ-4,311 ל-5,697. 1,386 נהגים עם הערך 999999, שהוא חסר מוסווה. החוסר עצמו מנבא עלות, ולכן נשמר ב-Pipeline בעזרת עמודת סימון חוסר.

vehicle_value: נקרא כטקסט בגלל פסיקים, ואחרי הניקוי הוא מספר.

</div>

<div dir="rtl">

### פונקציות הפיצ'רים

שתי פונקציות שמוסיפות עמודות חדשות אחרי הניקוי, בחוקים קבועים בלבד: add_driver_features של ליעד (שישה פיצ'רים) ו-add_vehicle_features של דניאל (שני פיצ'רים). החקירה, הבדיקות והנימוק לכל פיצ'ר נמצאים במחברות driver_features ו-vehicle_features, וטבלת הפיצ'רים בסעיף 4.

</div>

In [18]:
# פונקציית הפיצ'רים של עמודות הנהג: מקבלת טבלה אחרי הניקוי ומוסיפה שש עמודות חדשות בלי לשנות את הקיימות
DRIVER_FEATURES = ["age_sq", "is_young", "is_new_driver", "clean_ratio", "claims_x_violations", "log_density"]

def add_driver_features(df):
    df = df.copy()
    age = df["driver_age"]
    license_years = df["license_years"]

    # גיל בריבוע: מאפשר למודל הלינארי לתאר את צורת האות יו של הגיל
    df["age_sq"] = age ** 2

    # נהג צעיר: אחד עד גיל עשרים ואחת כולל, ואם הגיל חסר גם הדגל חסר
    df["is_young"] = (age <= 21).astype(float).where(age.notna())

    # נהג חדש: אחד אם הוותק אפס או אחת, ואם הוותק חסר גם הדגל חסר
    df["is_new_driver"] = (license_years <= 1).astype(float).where(license_years.notna())

    # יחס שנים נקיות: שנים ללא תביעה חלקי ותק, ורק כשהוותק חיובי כדי לא לחלק באפס
    df["clean_ratio"] = (df["no_claims_years"] / license_years).where(license_years > 0)

    # אינטראקציה: תביעות קודמות כפול דוחות, וחסר אם אחד מהם חסר
    df["claims_x_violations"] = df["prior_claims_5y"] * df["violations_3y"]

    # לוג של הצפיפות: רק לערך חיובי, וערך חסר נשאר חסר
    density = df["region_density"]
    df["log_density"] = np.log(density.where(density > 0))

    return df

In [19]:
# פונקציה שמוסיפה שתי עמודות חדשות לטבלה אחרי הניקוי בלי לשנות את העמודות הקיימות
VEHICLE_FEATURES = ["value_x_work", "young_x_hp"]

def add_vehicle_features(df):
    df = df.copy()
    usage = df["usage_type"]
    age = df["driver_age"]

    # שימוש לעבודה: אחד למשלוחים ולנהיגה מקצועית, ואם סוג השימוש חסר גם הדגל חסר
    is_work = usage.isin(["delivery", "professional"]).astype(float).where(usage.notna())

    # שווי הרכב כפול שימוש לעבודה, וחסר אם אחד מהם חסר
    df["value_x_work"] = df["vehicle_value"] * is_work

    # נהג צעיר עד גיל עשרים ואחת כולל, ואם הגיל חסר גם הדגל חסר
    is_young = (age <= 21).astype(float).where(age.notna())

    # עוצמת המנוע כפול נהג צעיר, וחסר אם אחד מהם חסר
    df["young_x_hp"] = df["engine_hp"] * is_young

    return df

<div dir="rtl">

### הפונקציה prepare_data

הפונקציה מנקה, מוסיפה את שמונת הפיצ'רים, ומחזירה רק את 30 העמודות שהמודל מקבל, תמיד באותו סדר: 22 עמודות מקוריות (RAW_COLS) ועוד 8 פיצ'רים.

היא בוחרת עמודות לפי רשימה של מה שמותר (FEATURE_COLS), ולא מוחקת לפי רשימה של מה שאסור. כך עמודה לא צפויה בקובץ של הבודק יורדת לבד, ועמודות ה-Leakage והמזהים לא נכנסות אף פעם.

אם עמודה מקורית חסרה בקלט, היא נוספת כריקה. ה-Pipeline משלים אותה, וכך הקוד לא קורס.

</div>

In [20]:
# רשימות העמודות המקוריות ועמודות המודל, ופונקציית ההכנה שמנקה מוסיפה פיצרים ומחזירה רק את עמודות המודל
TARGET = "annual_claim_cost"
RAW_COLS = [
    "country", "area_type", "region_density", "winter_severity", "driver_age", "gender", "marital_status",
    "license_years", "prior_claims_5y", "no_claims_years", "violations_3y",
    "vehicle_type", "vehicle_age", "engine_hp", "vehicle_value", "safety_features_count", "has_telematics",
    "usage_type", "annual_km_estimated", "pct_intercity", "pct_night_driving", "parking_type",
]
FEATURE_COLS = RAW_COLS + DRIVER_FEATURES + VEHICLE_FEATURES

def prepare_data(df):
    # עמודה מקורית חסרה נוספת כריקה, ניקוי, הוספת פיצרים, ושמירה רק על עמודות המודל בסדר קבוע
    df = df.copy()
    for c in RAW_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df = clean_driver_cols(df)
    df = clean_vehicle_cols(df)
    df = add_driver_features(df)
    df = add_vehicle_features(df)
    return df[FEATURE_COLS]

In [21]:
# הרצה על כל הדאטה ועל שורה בודדת: גודל, עמודות שירדו וערכים חסרים שנשארים ל-Pipeline
X = prepare_data(df)
print("Shape:", X.shape)
print("Removed columns:", sorted(set(df.columns) - set(X.columns)))
missing = X.isna().sum()
print(missing[missing > 0])
prepare_data(df.iloc[[0]]).T

Shape: (45000, 30)
Removed columns: ['agent_code', 'annual_claim_cost', 'at_fault', 'claim_count', 'current_premium', 'customer_id_hash', 'days_to_settlement', 'police_report_filed']
license_years           153
prior_claims_5y        1955
has_telematics          636
annual_km_estimated    5697
is_new_driver           153
clean_ratio            1340
claims_x_violations    1955
dtype: int64


,0
country,ES
area_type,suburban
region_density,855.0
winter_severity,1.3
driver_age,75.0
gender,F
marital_status,widowed
license_years,54.0
prior_claims_5y,2.0
no_claims_years,0.0


<div dir="rtl">

prepare_data מחזירה 45,000 שורות ו-30 עמודות: 22 מקוריות ו-8 פיצ'רים. שמונה עמודות הוסרו: חמש עמודות Leakage, שני מזהים ועמודת המטרה.

נשארים ערכים חסרים בשבע עמודות, וה-Pipeline ישלים אותם: license_years (153), prior_claims_5y (1,955), has_telematics (636), annual_km_estimated (5,697), ושלושה פיצ'רים שעמודת המקור שלהם חסרה: is_new_driver (153), clean_ratio (1,340) ו-claims_x_violations (1,955).

הפונקציה רצה גם על שורה בודדת, כמו שההנחיות דורשות.

</div>

<div dir="rtl">

### בדיקות קיצון ל-prepare_data

אם prepare_data קורסת על הקובץ של הבודק, מאבדים את כל נקודות התחרות. לכן מנסים לשבור אותה בכל קלט שעלול להגיע: שורה בודדת, שורה ריקה, ערכים מלוכלכים, קובץ שנקרא כטקסט, עמודות בסדר אחר ועמודה מיותרת.

</div>

In [22]:
# בדיקות קיצון לפונקציית ההכנה המלאה: ניסיון לשבור אותה בקלטים שעלולים להגיע בקובץ הבדיקה
import io
full = prepare_data(df)
checks = {}

# אלף וחמש מאות שורות בודדות צריכות לצאת זהות לתוצאה על כל הדאטה
idx = df.sample(1500, random_state=RANDOM_STATE).index
checks["1500 single rows match full run"] = all(prepare_data(df.loc[[i]]).equals(full.loc[[i]]) for i in idx)

# שורה שכולה ריקה כפי שהיא נקראת מקובץ
empty_csv = ",".join(df.columns) + "\n" + ",".join([""] * len(df.columns)) + "\n"
checks["empty row from file"] = bool(prepare_data(pd.read_csv(io.StringIO(empty_csv))).isna().all(axis=None))

# כל הקובץ נקרא כטקסט והתוצאה צריכה להיות זהה
as_text = prepare_data(pd.read_csv("dataset.csv", dtype=str))
num_cols = full.select_dtypes("number").columns
checks["whole file read as text"] = (
    np.allclose(as_text[num_cols].astype(float), full[num_cols].astype(float), equal_nan=True)
    and all(as_text[c].astype(object).equals(full[c].astype(object)) for c in full.columns if c not in num_cols))

# ערכים מלוכלכים בכל עמודה: מחרוזת ריקה, רווח, אינסוף כטקסט, חסר, ומספר עם פסיק
dirty = df.iloc[:5].copy().astype(object)
for c in FEATURE_COLS:
    dirty[c] = ["", " ", "inf", pd.NA, "1,200"]
try:
    out = prepare_data(dirty)
    checks["dirty values do not crash"] = True
    checks["no infinity after cleaning"] = not np.isinf(out.select_dtypes("number").astype(float)).any(axis=None)
except Exception as e:
    checks["dirty values do not crash"] = False
    print(type(e).__name__, e)

# עמודות טקסט שכולן ריקות
nan_text = df.iloc[:3].copy()
for c in ["country", "area_type", "gender", "marital_status", "vehicle_type", "usage_type", "parking_type"]:
    nan_text[c] = np.nan
try:
    prepare_data(nan_text)
    checks["all-empty text columns"] = True
except Exception as e:
    checks["all-empty text columns"] = False
    print(type(e).__name__, e)

# בלי עמודת המטרה, סדר עמודות מעורבב ועמודה מיותרת, כמו קובץ הבדיקה
test_like = df.drop(columns=TARGET).sample(frac=1, axis=1, random_state=RANDOM_STATE)
test_like["unexpected_column"] = 1
checks["no target, shuffled and extra columns"] = prepare_data(test_like).equals(full)

# אפס שורות, אינדקס כפול, הקלט לא משתנה, ושתי הרצות זהות
checks["zero rows"] = prepare_data(df.iloc[0:0]).shape == (0, len(FEATURE_COLS))
dup = df.iloc[:5].copy()
dup.index = [7] * 5
checks["duplicated index"] = prepare_data(dup).reset_index(drop=True).equals(full.iloc[:5].reset_index(drop=True))
before = df.copy()
prepare_data(before)
checks["input unchanged"] = before.equals(df)
checks["deterministic"] = prepare_data(df).equals(full)
checks["no leakage or id columns"] = not set(LEAKAGE_COLS + ID_COLS + [TARGET]) & set(full.columns)

for name, passed in checks.items():
    print("OK  " if passed else "FAIL", name)
print("Shape:", full.shape)

OK   1500 single rows match full run
OK   empty row from file
OK   whole file read as text
OK   dirty values do not crash
OK   no infinity after cleaning
OK   all-empty text columns
OK   no target, shuffled and extra columns
OK   zero rows
OK   duplicated index
OK   input unchanged
OK   deterministic
OK   no leakage or id columns
Shape: (45000, 30)


<div dir="rtl">

כל 12 הבדיקות עברו בסביבת הפרויקט (Python 3.12, pandas 3.1.0, scikit-learn 1.9.1), וגם ב-pandas 2.2. prepare_data לא קורסת על שורה בודדת, על שורה ריקה, על ערכים מלוכלכים, על קובץ שנקרא כטקסט ועל קובץ בלי עמודת המטרה, בסדר עמודות אחר ועם עמודה מיותרת. בכל המקרים היא מחזירה את אותה תוצאה כמו על הדאטה הרגיל.

</div>

<div dir="rtl">

### סימולציית בודק

מחקים את מה שהבודק יעשה: 500 נהגים בלי עמודת המטרה נשמרים לקובץ, נקראים ממנו מחדש, ועוברים דרך prepare_data. בשלב הזה עוד אין מודל, אז בודקים רק שהפלט תקין. את predict נוסיף כשה-Pipeline יהיה מוכן.

</div>

In [23]:
# סימולציית בודק: חמש מאות נהגים בלי עמודת המטרה נשמרים לקובץ ונקראים מחדש כמו קובץ הבדיקה
import os
sample = df.sample(500, random_state=RANDOM_STATE).drop(columns=TARGET)
sample.to_csv("grader_sim.csv", index=False)

df_test = pd.read_csv("grader_sim.csv")
X_test = prepare_data(df_test)

print("Shape:", X_test.shape)
print("Same columns and order:", list(X_test.columns) == FEATURE_COLS)
print("Same values as full run:", X_test.reset_index(drop=True).equals(full.loc[sample.index].reset_index(drop=True)))
os.remove("grader_sim.csv")

Shape: (500, 30)
Same columns and order: True
Same values as full run: True


<div dir="rtl">

הסימולציה עברה: 500 נהגים שנקראו מקובץ חיצוני, בלי עמודת המטרה, יצאו עם 30 העמודות בסדר הנכון, ועם ערכים זהים לאלה שעל כל הדאטה. כשה-Pipeline יהיה מוכן, נוסיף כאן את שלב ה-predict.

</div>